# 01. Тензоры и autograd: фундамент

**Проект:** решение MNIST на базовых тензорах PyTorch, без `torch.nn` и `torch.optim`.

Этот ноутбук - разогрев. В нем нет MNIST, зато есть все, на чем он потом держится:
что такое тензор, что делает `requires_grad`, что именно происходит в `loss.backward()`
и где новички теряют градиенты.

## План всего проекта

| Ноутбук | Что в нем |
|---|---|
| **01** (этот) | тензоры, вычислительный граф, autograd, численная проверка градиента |
| **02** | линейная модель 784 -> 10: softmax, перекрестная энтропия, ручной вывод градиентов, обучение (около 92%) |
| **03** | MLP 784 -> 128 -> 10: ReLU, цепное правило через скрытый слой, ручной backward (около 97%) |
| **04** | та же сеть через `nn.Module` и `optim.SGD` - что именно библиотека сделала за нас |

## Принцип, которого держимся везде

Градиенты в обучении считает `loss.backward()` - это нормально, так работают все.
Но **каждую** величину, которую выдает autograd, мы сначала выводим руками
на бумаге, пишем формулой из тензорных операций и проверяем на равенство.
Если ручная формула и autograd совпали до седьмого знака - значит математику
поняли правильно, а не "вроде заработало".

---
## 0. Запуск

Ноутбук самодостаточный: он не импортирует ничего из проекта и работает
в Google Colab без подготовки. Нужен только `torch`, а он в Colab уже стоит.

Локально в PyCharm: интерпретатор из `.venv`, зависимости из `requirements.txt`.

In [9]:
import torch

print("версия torch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

# Фиксируем генератор случайных чисел. Без этого каждый перезапуск ноутбука
# дает другие числа, и сравнивать результаты между запусками невозможно.
torch.manual_seed(0)

версия torch: 2.14.1
CUDA доступна: False


---
## 1. Тензор

Тензор - это многомерный массив чисел одного типа плюс две вещи, которых
нет у обычного списка Python:

1. все операции над ним выполняются сразу над всем массивом (векторизация),
2. он умеет запоминать, как его получили, чтобы потом посчитать производную.

Терминология по числу осей:

| Число осей | Название | Пример в нашей задаче |
|---|---|---|
| 0 | скаляр | значение ошибки `loss` |
| 1 | вектор | смещения `b`, форма (10,) |
| 2 | матрица | батч картинок (64, 784); веса `W` (784, 10) |
| 3 и более | тензор | картинки без разворачивания, (64, 1, 28, 28) |

Два свойства, за которыми надо следить постоянно: **форма** (`.shape`) и
**тип** (`.dtype`). Девять ошибок из десяти в таком коде - это несошедшиеся
формы или `float` там, где нужен `int64`.

In [10]:
# Из списка Python
v = torch.tensor([1.0, 2.0, 3.0])
print("v        =", v, "| форма", tuple(v.shape), "| тип", v.dtype)

# Матрица 2x3 из нулей
m = torch.zeros(2, 3)
print("m.shape  =", tuple(m.shape))

# Нормальный шум нужной формы
r = torch.randn(2, 3)
print("r        =\n", r)

# dtype выводится из данных: целые числа дают int64, а не float
i = torch.tensor([1, 2, 3])
print("i.dtype  =", i.dtype, "(метки классов должны быть именно такими)")

# Приведение типа
print("i.float()=", i.float(), i.float().dtype)

v        = tensor([1., 2., 3.]) | форма (3,) | тип torch.float32
m.shape  = (2, 3)
r        =
 tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986]])
i.dtype  = torch.int64 (метки классов должны быть именно такими)
i.float()= tensor([1., 2., 3.]) torch.float32


### Матричное умножение и broadcasting

Две операции, из которых состоит весь линейный слой.

**Матричное умножение** `A @ B`: формы должны стыковаться как
`(n, k) @ (k, m) -> (n, m)`, внутренние размерности обязаны совпасть.
Для нас это `(64, 784) @ (784, 10) -> (64, 10)`: батч из 64 картинок
превращается в 64 набора по 10 логитов.

**Broadcasting** - автоматическое растягивание меньшего тензора до формы
большего. В `z = x @ W + b` слагаемое `b` имеет форму (10,), а `x @ W` -
форму (64, 10). PyTorch мысленно повторяет `b` 64 раза и складывает
построчно. Копий в памяти при этом не создается.

Правило broadcasting читается с конца: оси сравниваются справа налево и
совместимы, если равны либо одна из них равна 1.

In [11]:
x = torch.randn(64, 784)   # батч картинок
W = torch.randn(784, 10)   # веса
b = torch.randn(10)        # смещения

z = x @ W + b
print("x @ W ->", tuple((x @ W).shape))
print("b     ->", tuple(b.shape), "растянулось до", tuple(z.shape))
print("z     ->", tuple(z.shape))

# Проверим, что broadcasting сделал именно то, что мы думаем:
# вручную посчитаем один элемент z[5, 3] как скалярное произведение
manual = (x[5] * W[:, 3]).sum() + b[3]
print(f"\nz[5,3] от PyTorch: {z[5, 3]:.6f}")
print(f"то же вручную:     {manual:.6f}")
print("совпало:", torch.allclose(z[5, 3], manual, atol=1e-5))

x @ W -> (64, 10)
b     -> (10,) растянулось до (64, 10)
z     -> (64, 10)

z[5,3] от PyTorch: 3.672184
то же вручную:     3.672193
совпало: True


Почему `torch.allclose`, а не `==`. Числа типа `float32` хранятся с точностью
примерно семь значащих десятичных цифр, и сложение их **неассоциативно**:
`(a + b) + c` и `a + (b + c)` могут отличаться в последних битах. PyTorch
внутри `@` складывает 784 слагаемых в своем порядке, мы в `.sum()` - в своем.
Поэтому точного равенства ждать нельзя, а расхождение порядка `1e-7` - норма.

Это важно забежав вперед: когда будем сверять ручные градиенты с autograd,
критерием станет не равенство, а `allclose`.

---
## 2. requires_grad и вычислительный граф

Теперь главное.

Когда у тензора стоит `requires_grad=True`, PyTorch начинает **записывать все
операции с ним** в структуру, которая называется вычислительным графом.
Граф - это запись о том, из чего получен каждый промежуточный результат и
какой операцией. Строится он на ходу, во время прямого прохода (поэтому
подход называется define-by-run).

Дальше `loss.backward()` идет по этому графу от конца к началу и применяет
цепное правило. Результат складывается в поле `.grad` у тех тензоров, которые
были помечены `requires_grad=True`.

Разберем на примере из ноутбука преподавателя:

$$f(x, y) = (x + y)^2 + 2xy, \qquad x = 2, \quad y = -4$$

Считаем руками. Раскроем скобки:

$$f = x^2 + 2xy + y^2 + 2xy = x^2 + 4xy + y^2$$

$$\frac{\partial f}{\partial x} = 2x + 4y = 2 \cdot 2 + 4 \cdot (-4) = 4 - 16 = -12$$

$$\frac{\partial f}{\partial y} = 4x + 2y = 4 \cdot 2 + 2 \cdot (-4) = 8 - 8 = 0$$

Проверим, что autograd выдаст то же самое.

In [12]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(-4.0, requires_grad=True)

f = (x + y) ** 2 + 2 * x * y

print("f =", f.item(), "  (ожидаем (2-4)^2 + 2*2*(-4) = 4 - 16 = -12)")
print("f.requires_grad =", f.requires_grad, "- результат тоже в графе")
print("f.grad_fn =", f.grad_fn, "- операция, которой f получен")

f.backward()   # проход по графу назад

print("\nautograd:      df/dx =", x.grad.item(), "  df/dy =", y.grad.item())
print("наши формулы:  df/dx = -12.0   df/dy = 0.0")

assert x.grad.item() == -12.0 and y.grad.item() == 0.0, "формулы разошлись"
print("\nсовпало")

f = -12.0   (ожидаем (2-4)^2 + 2*2*(-4) = 4 - 16 = -12)
f.requires_grad = True - результат тоже в графе
f.grad_fn = <AddBackward0 object at 0x10fe22140> - операция, которой f получен

autograd:      df/dx = -12.0   df/dy = 0.0
наши формулы:  df/dx = -12.0   df/dy = 0.0

совпало


### Как выглядит сам граф

У каждого тензора, полученного вычислением, есть `.grad_fn` - узел графа.
У узла есть `.next_functions` - на кого он ссылается дальше. Пройдемся по
этой структуре и напечатаем ее: для `f = (x+y)**2 + 2*x*y` должно получиться
дерево со сложением в корне, возведением в степень и умножениями в ветвях.

`AccumulateGrad` в листьях - это и есть узлы, которые пишут результат
в `.grad` у `x` и `y`.

In [13]:
def print_graph(node, depth=0):
    # Рекурсивный обход графа от корня к листьям
    if node is None:
        return
    print("   " * depth + "+- " + type(node).__name__)
    for nxt, _ in getattr(node, "next_functions", ()):
        print_graph(nxt, depth + 1)

x2 = torch.tensor(2.0, requires_grad=True)
y2 = torch.tensor(-4.0, requires_grad=True)
f2 = (x2 + y2) ** 2 + 2 * x2 * y2

print("граф для f = (x+y)**2 + 2*x*y:\n")
print_graph(f2.grad_fn)

граф для f = (x+y)**2 + 2*x*y:

+- AddBackward0
   +- PowBackward0
      +- AddBackward0
         +- AccumulateGrad
         +- AccumulateGrad
   +- MulBackward0
      +- MulBackward0
         +- AccumulateGrad
      +- AccumulateGrad


Читается снизу вверх, и это ровно порядок обратного прохода:
`AddBackward0` (корень, сложение двух слагаемых) получает градиент 1,
раздает его в `PowBackward0` и `MulBackward0`, те дальше по цепочке,
а в самом низу `AccumulateGrad` кладет итог в `x.grad` и `y.grad`.

То же самое мы будем делать руками в ноутбуках 02 и 03, только граф там
будет не из скаляров, а из матриц.

---
## 3. Три грабли, на которых теряются градиенты

### 3.1 `.grad` не перезаписывается, а НАКАПЛИВАЕТСЯ

`backward()` не кладет новое значение в `.grad`, а **прибавляет** к тому, что
там лежало. Так сделано специально: если один параметр используется в графе
несколько раз, его вклады должны сложиться.

Побочный эффект: если в цикле обучения забыть обнулить `.grad`, там накопится
сумма градиентов по всем прошедшим батчам. Шаги станут все больше, и ошибка
уйдет в `NaN` за несколько десятков итераций. Это самая частая ошибка
в ручном цикле обучения.

In [14]:
w = torch.tensor(3.0, requires_grad=True)

for i in range(1, 4):
    loss = w ** 2          # d(w^2)/dw = 2w = 6
    loss.backward()
    print(f"вызов {i}: w.grad = {w.grad.item():>5.1f}  (каждый раз прибавилось 6)")

print("\nтеперь с обнулением:")
w.grad.zero_()
for i in range(1, 4):
    loss = w ** 2
    loss.backward()
    print(f"вызов {i}: w.grad = {w.grad.item():>5.1f}")
    w.grad.zero_()

вызов 1: w.grad =   6.0  (каждый раз прибавилось 6)
вызов 2: w.grad =  12.0  (каждый раз прибавилось 6)
вызов 3: w.grad =  18.0  (каждый раз прибавилось 6)

теперь с обнулением:
вызов 1: w.grad =   6.0
вызов 2: w.grad =   6.0
вызов 3: w.grad =   6.0


### 3.2 Обновление веса должно идти под `torch.no_grad()`

Шаг спуска `w -= lr * w.grad` - это тоже операция над тензором с
`requires_grad=True`. Если не отключить запись графа, то:

* новое значение `w` окажется вычисленным **из старого**, то есть попадет в граф;
* граф будет расти с каждым шагом обучения, память кончится;
* PyTorch вообще запретит такую операцию над листовым тензором.

`torch.no_grad()` - это контекст, внутри которого граф не строится вообще.
Используется в двух местах: шаг оптимизатора и любая оценка модели
(при подсчете точности градиенты не нужны).

In [15]:
w = torch.tensor(3.0, requires_grad=True)
loss = w ** 2
loss.backward()

# Так - ошибка:
try:
    w -= 0.1 * w.grad
except RuntimeError as e:
    print("БЕЗ no_grad:", str(e)[:110], "...")

# Так - правильно:
with torch.no_grad():
    w -= 0.1 * w.grad
    w.grad.zero_()

print("\nС no_grad: w =", w.item(), "(было 3.0, градиент 6.0, шаг 0.1 -> 3.0 - 0.6)")
print("w.requires_grad =", w.requires_grad, "- параметр остался обучаемым")

БЕЗ no_grad: a leaf Variable that requires grad is being used in an in-place operation. ...

С no_grad: w = 2.4000000953674316 (было 3.0, градиент 6.0, шаг 0.1 -> 3.0 - 0.6)
w.requires_grad = True - параметр остался обучаемым


### 3.3 `.item()`, `float()` и `.detach()`

Пока мы держим ссылку на тензор из графа, живет весь граф, и память не
освобождается. Поэтому, когда от тензора нужно только число (записать ошибку
в историю, напечатать), берут `.item()` или `float(t.detach())`, а не сам тензор.

`.detach()` возвращает тот же тензор, но вырезанный из графа: значения те же,
память общая, а история вычислений потеряна. Это нужно и нам в формулах
ручных градиентов: сама проверка не должна попадать в граф.

In [16]:
a = torch.tensor([1.0, 2.0], requires_grad=True)
s = (a ** 2).sum()

print("s                       :", s, "<- тензор, тянет за собой граф")
print("s.item()                :", s.item(), "<- обычное число Python")
print("s.detach()              :", s.detach(), "<- тензор без графа")
print("s.detach().requires_grad:", s.detach().requires_grad)

# detach() делит память с исходным тензором
d = a.detach()
d[0] = 99.0
print("\nпосле d[0]=99 исходный a =", a.data, "- память общая, это не копия")
print("для настоящей копии нужен a.detach().clone()")

s                       : tensor(5., grad_fn=<SumBackward0>) <- тензор, тянет за собой граф
s.item()                : 5.0 <- обычное число Python
s.detach()              : tensor(5.) <- тензор без графа
s.detach().requires_grad: False

после d[0]=99 исходный a = tensor([99.,  2.]) - память общая, это не копия
для настоящей копии нужен a.detach().clone()


---
## 4. Численная проверка градиента

Как убедиться, что формула производной верна, если проверять нечем?
Посчитать производную численно, по определению:

$$\frac{\partial f}{\partial w} \approx \frac{f(w + h) - f(w - h)}{2h}$$

Это **центральная разностная схема**. Односторонняя
$\big(f(w+h) - f(w)\big) / h$ тоже работает, но у нее погрешность порядка $h$,
а у центральной - порядка $h^2$, то есть на том же $h$ она точнее на порядки.

Выбор $h$ - компромисс: слишком большой дает ошибку аппроксимации, слишком
маленький - потерю точности при вычитании почти равных чисел. Для `float64`
хорошо работает $h \approx 10^{-5}$.

Этот прием - независимый арбитр: он не знает ни про autograd, ни про наши
формулы. Если все три источника сошлись, ошибка исключена.

In [17]:
def numeric_grad(fn, w, h=1e-5):
    # Численный градиент функции fn по тензору w (центральная схема).
    #
    # Для каждого элемента w по очереди: сдвинуть на +h, посчитать fn,
    # сдвинуть на -h, посчитать fn, взять разность, поделить на 2h.
    # Стоит 2*N вычислений fn, где N - число элементов, поэтому для
    # настоящей сети так считают только на крошечном примере.
    w = w.detach().to(torch.float64)
    grad = torch.zeros_like(w)
    flat_w, flat_g = w.view(-1), grad.view(-1)

    for i in range(flat_w.numel()):
        original = flat_w[i].item()

        flat_w[i] = original + h
        f_plus = float(fn(w))

        flat_w[i] = original - h
        f_minus = float(fn(w))

        flat_w[i] = original                 # вернуть значение на место
        flat_g[i] = (f_plus - f_minus) / (2 * h)

    return grad


# Проверим на функции, у которой производная известна аналитически:
#   g(w) = sum(w^3) - 2*sum(w),   dg/dw_i = 3*w_i^2 - 2
def g(w):
    return (w ** 3).sum() - 2 * w.sum()

w0 = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=torch.float64)

analytic = 3 * w0 ** 2 - 2                                  # формула на бумаге
numeric = numeric_grad(g, w0)                               # численно

w_auto = w0.clone().requires_grad_()                        # autograd
g(w_auto).backward()

print("аналитически:", analytic.tolist())
print("численно:    ", [round(v, 8) for v in numeric.tolist()])
print("autograd:    ", w_auto.grad.tolist())
print()
print("аналитика vs численно:", torch.allclose(analytic, numeric, atol=1e-6))
print("аналитика vs autograd:", torch.allclose(analytic, w_auto.grad, atol=1e-10))

аналитически: [1.0, 10.0, -1.25, 25.0]
численно:     [1.0, 10.0, -1.25, 25.0]
autograd:     [1.0, 10.0, -1.25, 25.0]

аналитика vs численно: True
аналитика vs autograd: True


---
## 5. Градиент матричной функции: разминка перед MNIST

В MNIST все производные берутся не по скаляру, а по матрице весов. Соберем
на маленьком примере то, что в ноутбуке 02 станет центральной формулой.

Пусть
$$z = x W, \qquad L = \tfrac{1}{2}\sum_{n,k} z_{n,k}^2$$

Поэлементно $z_{n,k} = \sum_i x_{n,i} W_{i,k}$, и значит
$\partial L / \partial z_{n,k} = z_{n,k}$.

Вес $W_{i,k}$ входит в $z_{n,k}$ для каждого примера $n$, причем
$\partial z_{n,k} / \partial W_{i,k} = x_{n,i}$. Складываем вклады по всем $n$:

$$\frac{\partial L}{\partial W_{i,k}} = \sum_n \frac{\partial L}{\partial z_{n,k}} \, x_{n,i}$$

Справа - это в точности элемент $[i,k]$ произведения $x^\top G$, где
$G = \partial L / \partial z$. Итого:

$$\frac{\partial L}{\partial W} = x^\top G$$

**Прием, который экономит часы отладки:** порядок множителей можно не
выводить, а восстановить по формам. Нужна матрица формы $W$, то есть $(D, C)$.
Есть $x$ формы $(B, D)$ и $G$ формы $(B, C)$. Из них $(D, C)$ собирается
единственным способом: $(D,B) \times (B,C)$, то есть `x.T @ G`.

In [18]:
B, D, C = 5, 4, 3   # батч, признаки, классы

x = torch.randn(B, D)
W = torch.randn(D, C, requires_grad=True)

z = x @ W
L = 0.5 * (z ** 2).sum()
L.backward()

# Ручная формула
G = z.detach()              # dL/dz = z для L = 0.5 * sum(z^2)
grad_manual = x.T @ G       # dL/dW = x^T @ G

print("формы:  x", tuple(x.shape), " G", tuple(G.shape),
      " ->  x.T @ G", tuple(grad_manual.shape))
print("W.grad        ", tuple(W.grad.shape))
print()
print("max |ручное - autograd| =", (grad_manual - W.grad).abs().max().item())
print("совпало:", torch.allclose(grad_manual, W.grad, atol=1e-5))

формы:  x (5, 4)  G (5, 3)  ->  x.T @ G (4, 3)
W.grad         (4, 3)

max |ручное - autograd| = 0.0
совпало: True


---
## Итоги ноутбука 01

**Что усвоили:**

1. Тензор - массив с векторизованными операциями и памятью о том, как он получен.
   Следить всегда за `.shape` и `.dtype`.
2. `requires_grad=True` включает запись вычислительного графа. `backward()` идет
   по графу назад и кладет результат в `.grad`.
3. `.grad` накапливается - обнулять обязательно.
4. Шаг обновления веса и любая оценка модели - под `torch.no_grad()`.
5. `float32` неассоциативен, поэтому сверка градиентов - через `allclose`,
   а не через `==`.
6. Формулу градиента можно проверить тремя независимыми способами: аналитика,
   численная разность, autograd. Все три обязаны сойтись.
7. Для линейного слоя $\partial L/\partial W = x^\top G$, и порядок множителей
   восстанавливается по формам.

**Дальше:** ноутбук `02_linear_softmax_manual.ipynb` - MNIST, softmax,
перекрестная энтропия и первая обученная модель.